In [1]:
import numpy as np

In [2]:
import geopandas as gpd
import folium
from folium import plugins
import pickle
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Configuración de directorios de cache
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

# Definir rutas de cache para cada capa de datos
CACHE_PATHS = {
    'centros_salud': CACHE_DIR / 'centros_salud.pkl',
    'hospitales': CACHE_DIR / 'hospitales.pkl',
    'farmacias': CACHE_DIR / 'farmacias.pkl',
    'zonas_verdes': CACHE_DIR / 'zonas_verdes.pkl',
    'poblaciones': CACHE_DIR / 'poblaciones.pkl',
}

print(f"Directorio de cache: {CACHE_DIR.absolute()}")

Directorio de cache: d:\proyectos_bootcamp\proyecto final\Andaluc-a-Quick-Data\cache


In [4]:
def guardar_en_cache(gdf, nombre_capa):
    """Guardar un GeoDataFrame en cache usando pickle"""
    cache_path = CACHE_PATHS[nombre_capa]
    with open(cache_path, 'wb') as f:
        pickle.dump(gdf, f)
    print(f"✓ Datos de {nombre_capa} guardados en cache: {cache_path}")

def cargar_del_cache(nombre_capa):
    """Cargar un GeoDataFrame del cache"""
    cache_path = CACHE_PATHS[nombre_capa]
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            gdf = pickle.load(f)
        print(f"✓ Datos de {nombre_capa} cargados del cache")
        return gdf
    else:
        print(f"✗ Cache no encontrado para {nombre_capa}")
        return None

def listar_cache_disponible():
    """Listar todos los datasets que están en cache"""
    disponibles = {}
    for nombre, path in CACHE_PATHS.items():
        if path.exists():
            disponibles[nombre] = path
    return disponibles

In [5]:
def extraer_y_guardar_datos():
    """Extrae datos de WFS de IDEAndalucía y los guarda en cache"""
    
    # Datos de centros de salud
    print("Extrayendo Centros de Salud...")
    try:
        url = "https://www.ideandalucia.es/services/DERA_g12_servicios/wfs?service=wfs&request=getcapabilities"
        gdf = gpd.read_file(url, layer='DERA_g12_servicios:g12_01_CentroSalud', engine="pyogrio", encoding="utf-8")
        gdf = gdf.drop(columns=['gml_id', 'id_dera', 'nombre', 'direccion', 'telefono', 'web', 'localidad', 'cod_mun'], errors='ignore')
        guardar_en_cache(gdf, 'centros_salud')
    except Exception as e:
        print(f"Error al extraer centros de salud: {e}")
    
    # Datos de hospitales
    print("Extrayendo Hospitales...")
    try:
        gdf = gpd.read_file(url, layer='DERA_g12_servicios:g12_02_Hospital_CAE', engine="pyogrio", encoding="utf-8")
        gdf = gdf.drop(columns=['gml_id', 'id_dera', 'web', 'cod_mun'], errors='ignore')
        guardar_en_cache(gdf, 'hospitales')
    except Exception as e:
        print(f"Error al extraer hospitales: {e}")
    
    # Datos de farmacias
    print("Extrayendo Farmacias...")
    try:
        gdf = gpd.read_file(url, layer='DERA_g12_servicios:g12_04_Farmacia', engine="pyogrio", encoding="utf-8")
        gdf = gdf.drop(columns=['gml_id', 'id_dera', 'nica', 'cod_mun'], errors='ignore')
        guardar_en_cache(gdf, 'farmacias')
    except Exception as e:
        print(f"Error al extraer farmacias: {e}")
    
    # Datos de zonas verdes
    print("Extrayendo Zonas Verdes...")
    try:
        url2 = "https://www.ideandalucia.es/services/DERA_g7_sistema_urbano/wfs?service=wfs&request=getcapabilities"
        gdf = gpd.read_file(url2, layer='DERA_g7_sistema_urbano:g07_06_ZonaVerde', engine="pyogrio", encoding="utf-8")
        gdf = gdf.drop(columns=['gml_id', 'id_dera', 'nombre'], errors='ignore')
        guardar_en_cache(gdf, 'zonas_verdes')
    except Exception as e:
        print(f"Error al extraer zonas verdes: {e}")
    
    # Datos de poblaciones
    print("Extrayendo Poblaciones...")
    try:
        gdf = gpd.read_file(url2, layer='DERA_g7_sistema_urbano:g07_01_Poblaciones', engine="pyogrio", encoding="utf-8")
        gdf = gdf.drop(columns=['gml_id', 'id_dera', 'cod_sipob', 'pob_ine', 'nombre', 'nivel', 'cod_mun'], errors='ignore')
        guardar_en_cache(gdf, 'poblaciones')
    except Exception as e:
        print(f"Error al extraer poblaciones: {e}")
    
    print("\n✓ Extracción completada")

# Ejecutar la extracción si el cache está vacío
cache_disponible = listar_cache_disponible()
if len(cache_disponible) == 0:
    print("Cache vacío. Extrayendo datos de IDEAndalucía...")
    extraer_y_guardar_datos()
else:
    print(f"Cache disponible con {len(cache_disponible)} dataset(s): {list(cache_disponible.keys())}")

Cache vacío. Extrayendo datos de IDEAndalucía...
Extrayendo Centros de Salud...
✓ Datos de centros_salud guardados en cache: cache\centros_salud.pkl
Extrayendo Hospitales...
✓ Datos de hospitales guardados en cache: cache\hospitales.pkl
Extrayendo Farmacias...
✓ Datos de farmacias guardados en cache: cache\farmacias.pkl
Extrayendo Zonas Verdes...
✓ Datos de zonas_verdes guardados en cache: cache\zonas_verdes.pkl
Extrayendo Poblaciones...
✓ Datos de poblaciones guardados en cache: cache\poblaciones.pkl

✓ Extracción completada


## Cargar datos del cache

In [6]:
# Cargar todos los datos del cache
datos_cache = {}
for nombre in CACHE_PATHS.keys():
    gdf = cargar_del_cache(nombre)
    if gdf is not None:
        datos_cache[nombre] = gdf
        print(f"  - {nombre}: {len(gdf)} registros")

print(f"\nTotal de capas cargadas: {len(datos_cache)}")

✓ Datos de centros_salud cargados del cache
  - centros_salud: 1516 registros
✓ Datos de hospitales cargados del cache
  - hospitales: 183 registros
✓ Datos de farmacias cargados del cache
  - farmacias: 3874 registros
✓ Datos de zonas_verdes cargados del cache
  - zonas_verdes: 9279 registros
✓ Datos de poblaciones cargados del cache
  - poblaciones: 57619 registros

Total de capas cargadas: 5


## Mapa Base de Andalucía
Creación de un mapa interactivo con todas las capas de datos disponibles.

In [ ]:
# Calcular el centro de Andalucía basado en los datos disponibles
if datos_cache:
    # Usar centros_salud como referencia
    bounds = datos_cache['centros_salud'].total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
else:
    # Coordenadas aproximadas de Andalucía
    center_lat = 37.5
    center_lon = -3.75

print(f"Centro de mapa: ({center_lat:.4f}, {center_lon:.4f})")

# Crear mapa base
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=8,
    tiles='OpenStreetMap'
)

# Colores para diferentes capas
colores = {
    'centros_salud': 'blue',
    'hospitales': 'red',
    'farmacias': 'green',
    'zonas_verdes': 'lightgreen',
    'poblaciones': 'purple'
}

# Agregar cada capa como FeatureGroup para control de capas
print("\nAgregando capas al mapa...")
for nombre, gdf in datos_cache.items():
    fg = folium.FeatureGroup(name=nombre.replace('_', ' ').title(), show=(nombre != 'poblaciones'))
    
    color = colores.get(nombre, 'gray')
    
    # Limitar elementos para no sobrecargar el mapa
    max_elementos = 500 if nombre == 'poblaciones' else 5000
    gdf_limitado = gdf.head(max_elementos) if len(gdf) > max_elementos else gdf
    
    print(f"  {nombre}: {len(gdf_limitado)} elementos agregados")
    
    # Agregar puntos/geometrías al mapa
    for idx, row in gdf_limitado.iterrows():
        try:
            geometry = row.geometry
            
            if geometry.geom_type == 'Point':
                folium.CircleMarker(
                    location=[geometry.y, geometry.x],
                    radius=5,
                    popup=f"{nombre.replace('_', ' ').title()}",
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.7
                ).add_to(fg)
            elif geometry.geom_type in ['Polygon', 'MultiPolygon']:
                if geometry.geom_type == 'MultiPolygon':
                    for poly in geometry.geoms:
                        coords = [[y, x] for x, y in poly.exterior.coords]
                        folium.Polygon(
                            coords,
                            color=color,
                            fill=True,
                            fillColor=color,
                            fillOpacity=0.3
                        ).add_to(fg)
                else:
                    coords = [[y, x] for x, y in geometry.exterior.coords]
                    folium.Polygon(
                        coords,
                        color=color,
                        fill=True,
                        fillColor=color,
                        fillOpacity=0.3
                    ).add_to(fg)
        except Exception as e:
            pass
    
    fg.add_to(m)

# Agregar control de capas
folium.LayerControl().add_to(m)

print("\n✓ Mapa base creado exitosamente")
# Mostrar el mapa
m

Centro de mapa: (4132265.7863, 360692.7576)


KeyboardInterrupt: 

## Mapas por Tipo de Servicio
Visualizaciones individuales para cada tipo de servicio o infraestructura.

In [ ]:
def crear_mapa_capa(nombre_capa, gdf):
    """Crear un mapa individual para una capa específica"""
    
    if gdf is None or len(gdf) == 0:
        print(f"No hay datos disponibles para {nombre_capa}")
        return None
    
    # Calcular centro
    bounds = gdf.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2
    
    # Crear mapa
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=8,
        tiles='CartoDB positron'
    )
    
    # Agregar datos al mapa
    color = colores.get(nombre_capa, 'gray')
    
    for idx, row in gdf.iterrows():
        geometry = row.geometry
        
        try:
            if geometry.geom_type == 'Point':
                # Crear popup con información disponible
                popup_text = f"<b>{nombre_capa.replace('_', ' ').title()}</b><br>"
                for col in gdf.columns:
                    if col != 'geometry' and not col.startswith('gml_'):
                        popup_text += f"{col}: {row[col]}<br>"
                
                folium.CircleMarker(
                    location=[geometry.y, geometry.x],
                    radius=6,
                    popup=folium.Popup(popup_text, max_width=300),
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.8,
                    weight=2
                ).add_to(m)
            
            elif geometry.geom_type in ['Polygon', 'MultiPolygon']:
                if geometry.geom_type == 'MultiPolygon':
                    for poly in geometry.geoms:
                        coords = [[y, x] for x, y in poly.exterior.coords]
                        folium.Polygon(
                            coords,
                            color=color,
                            fill=True,
                            fillColor=color,
                            fillOpacity=0.4,
                            weight=1
                        ).add_to(m)
                else:
                    coords = [[y, x] for x, y in geometry.exterior.coords]
                    folium.Polygon(
                        coords,
                        color=color,
                        fill=True,
                        fillColor=color,
                        fillOpacity=0.4,
                        weight=1
                    ).add_to(m)
        except Exception as e:
            print(f"Error procesando geometría: {e}")
            continue
    
    # Agregar cluster
    folium.plugins.HeatMap(
        [[row.geometry.y, row.geometry.x] for idx, row in gdf.iterrows() if row.geometry.geom_type == 'Point'],
        radius=15,
        blur=25,
        max_zoom=1
    ).add_to(m)
    
    # Ajustar vista a los datos
    try:
        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    except:
        pass
    
    return m

# Mapas por capa
print(f"Capas disponibles para mapas individuales: {list(datos_cache.keys())}\n")

### Mapa de Centros de Salud

In [ ]:
if 'centros_salud' in datos_cache:
    mapa_centros = crear_mapa_capa('centros_salud', datos_cache['centros_salud'])
    if mapa_centros:
        print(f"Mapa de Centros de Salud - {len(datos_cache['centros_salud'])} puntos")
        mapa_centros
else:
    print("No hay datos de centros de salud en el cache")

### Mapa de Hospitales

In [ ]:
if 'hospitales' in datos_cache:
    mapa_hospitales = crear_mapa_capa('hospitales', datos_cache['hospitales'])
    if mapa_hospitales:
        print(f"Mapa de Hospitales - {len(datos_cache['hospitales'])} puntos")
        mapa_hospitales
else:
    print("No hay datos de hospitales en el cache")

### Mapa de Farmacias

In [ ]:
if 'farmacias' in datos_cache:
    mapa_farmacias = crear_mapa_capa('farmacias', datos_cache['farmacias'])
    if mapa_farmacias:
        print(f"Mapa de Farmacias - {len(datos_cache['farmacias'])} puntos")
        mapa_farmacias
else:
    print("No hay datos de farmacias en el cache")

### Mapa de Zonas Verdes

In [ ]:
if 'zonas_verdes' in datos_cache:
    mapa_zonas_verdes = crear_mapa_capa('zonas_verdes', datos_cache['zonas_verdes'])
    if mapa_zonas_verdes:
        print(f"Mapa de Zonas Verdes - {len(datos_cache['zonas_verdes'])} áreas")
        mapa_zonas_verdes
else:
    print("No hay datos de zonas verdes en el cache")

## Exportar Mapas a HTML

In [ ]:
# Crear directorio para mapas HTML
MAPAS_DIR = Path("mapas_html")
MAPAS_DIR.mkdir(exist_ok=True)

def guardar_mapa_html(mapa, nombre_archivo):
    """Guardar un mapa folium como archivo HTML"""
    ruta = MAPAS_DIR / f"{nombre_archivo}.html"
    mapa.save(str(ruta))
    print(f"✓ Mapa guardado: {ruta.absolute()}")
    return ruta

# Guardar mapa base
print("Guardando mapas a HTML...\n")
guardar_mapa_html(m, "mapa_completo_andalucia")

# Guardar mapas individuales
if 'centros_salud' in datos_cache:
    mapa_centros = crear_mapa_capa('centros_salud', datos_cache['centros_salud'])
    guardar_mapa_html(mapa_centros, "mapa_centros_salud")

if 'hospitales' in datos_cache:
    mapa_hospitales = crear_mapa_capa('hospitales', datos_cache['hospitales'])
    guardar_mapa_html(mapa_hospitales, "mapa_hospitales")

if 'farmacias' in datos_cache:
    mapa_farmacias = crear_mapa_capa('farmacias', datos_cache['farmacias'])
    guardar_mapa_html(mapa_farmacias, "mapa_farmacias")

if 'zonas_verdes' in datos_cache:
    mapa_zonas_verdes = crear_mapa_capa('zonas_verdes', datos_cache['zonas_verdes'])
    guardar_mapa_html(mapa_zonas_verdes, "mapa_zonas_verdes")

print(f"\n✓ Todos los mapas guardados en: {MAPAS_DIR.absolute()}")

## Estadísticas de los Datos
Análisis estadístico de las capas de datos cargadas desde cache.

In [ ]:
import pandas as pd

# Crear tabla de estadísticas
print("=" * 60)
print("ESTADÍSTICAS DE CAPAS DE DATOS DESDE CACHE")
print("=" * 60)

estadisticas = []
for nombre, gdf in datos_cache.items():
    info = {
        'Capa': nombre.replace('_', ' ').title(),
        'Registros': len(gdf),
        'Geometría': gdf.geometry.type.unique()[0] if len(gdf) > 0 else 'N/A',
        'CRS': gdf.crs if gdf.crs else 'N/A',
        'Columnas': len(gdf.columns)
    }
    estadisticas.append(info)

df_stats = pd.DataFrame(estadisticas)
print("\n")
print(df_stats.to_string(index=False))
print("\n" + "=" * 60)
print(f"TOTAL: {sum(df['Registros'] for _, df in datos_cache.items())} elementos en {len(datos_cache)} capas")
print("=" * 60)

## Información y Referencia
### Estructura del Proyecto
- **cache/**: Directorio donde se almacenan los datos limpios en formato pickle
- **mapas_html/**: Directorio donde se guardan los mapas interactivos en HTML

### Cómo usar este notebook

1. **Extracción de datos** (Primera ejecución):
   - Si el cache está vacío, ejecuta la celda "Cargar y cachear datos desde Cleaning.ipynb"
   - Los datos se extraerán del WFS de IDEAndalucía y se guardarán en cache

2. **Uso del cache** (Ejecuciones posteriores):
   - Los datos se cargarán automáticamente del cache
   - Mucho más rápido que extraer desde el WFS cada vez

3. **Visualización de mapas**:
   - El mapa completo muestra todas las capas superpuestas
   - Los mapas individuales muestran cada capa por separado
   - Todos los mapas se guardan en la carpeta `mapas_html/`

### Capas disponibles
- **Centros de Salud**: Puntos de servicios médicos
- **Hospitales**: Hospitales y centros CAE
- **Farmacias**: Ubicaciones de farmacias
- **Zonas Verdes**: Parques y espacios verdes
- **Poblaciones**: Núcleos de población

### Funciones útiles
```python
# Cargar datos del cache
gdf = cargar_del_cache('centros_salud')

# Guardar datos en cache
guardar_en_cache(gdf_nuevo, 'nombre_capa')

# Crear un mapa individual
mapa = crear_mapa_capa('hospitales', datos_cache['hospitales'])

# Guardar mapa como HTML
guardar_mapa_html(mapa, 'mi_mapa')
```